# 03 Inference

Load the base model in 4-bit, attach the LoRA adapter, and run evaluation-style prompts (`docs/claude-3.md` Step 8). Requires CUDA.

In [ ]:
from pathlib import Path
import os


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "configs" / "qlora_config.yaml").is_file():
            return cand
    raise FileNotFoundError("Could not find configs/qlora_config.yaml")


REPO = find_repo_root()
os.chdir(REPO)
print("REPO:", REPO)

In [ ]:
from typing import Any

import torch
import yaml
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available()

with open("configs/qlora_config.yaml", encoding="utf-8") as fh:
    cfg: dict[str, Any] = yaml.safe_load(fh)

model_cfg = cfg["model"]
train_cfg = cfg["training"]
data_cfg = cfg["data"]

dtype_map = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}
compute_dtype = dtype_map[model_cfg["bnb_4bit_compute_dtype"]]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=model_cfg["load_in_4bit"],
    bnb_4bit_quant_type=model_cfg["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=model_cfg["bnb_4bit_use_double_quant"],
)

model_id = model_cfg["model_id"]
adapter_dir = Path(train_cfg["output_dir"]) / "adapter"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=model_cfg["trust_remote_code"],
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=model_cfg["trust_remote_code"],
)
model = PeftModel.from_pretrained(base, str(adapter_dir), torch_dtype=compute_dtype)
model.eval()
print("Loaded adapter from", adapter_dir)

In [ ]:
def build_chat_prompt(system_prompt: str, instruction: str, user_input: str) -> str:
    return (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{instruction}\n{user_input}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


@torch.no_grad()
def generate(instruction: str, user_input: str = "", max_new_tokens: int = 256) -> str:
    sp = data_cfg["system_prompt"]
    prompt = build_chat_prompt(sp, instruction, user_input)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=False)
    if "<|im_start|>assistant" in text:
        text = text.split("<|im_start|>assistant")[-1]
    for stop in ("<|im_end|>", "<|endoftext|>"):
        if stop in text:
            text = text.split(stop)[0]
    return text.strip()

In [ ]:
import sys

sys.path.insert(0, str(REPO / "scripts"))
from build_instructions import TYPED_TEMPLATES

EVAL_PROMPTS: list[tuple[str, str]] = [(t, "") for t in TYPED_TEMPLATES]

In [ ]:
import json

results = []
for instruction, ctx in EVAL_PROMPTS:
    out_text = generate(instruction, ctx)
    results.append({"instruction": instruction, "input": ctx, "output": out_text, "score": None})
    print("=" * 40)
    print(out_text[:500], "…" if len(out_text) > 500 else "")

out_path = Path("eval_results.jsonl")
with out_path.open("w", encoding="utf-8") as fh:
    for r in results:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
print("Wrote", out_path)